In [1]:
import os
import sys
import torch
import pandas as pd

PROJECT_ROOT = os.path.abspath("..")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.environment import HotelPricingEnv
from src.dqn_agent import DQNAgent
from src.config import TOTAL_ROOMS

In [2]:
env = HotelPricingEnv()

state_size = env.observation_space.shape[0]
action_size = env.action_space.n

agent = DQNAgent(state_size, action_size)

agent.model.load_state_dict(torch.load("../models/dqn_model.pth"))

agent.model.eval()

agent.epsilon = 0

In [3]:
trajectory = []

state, info = env.reset()
done = False
day_number = 1
total_revenue = 0

while not done:
    action = agent.select_action(state)
    next_state, reward, done, _, info = env.step(action)
    total_revenue += reward
    occupancy = ((TOTAL_ROOMS - env.rooms) / TOTAL_ROOMS) * 100

    trajectory.append({
        "Day": day_number,
        "Price": info["price"],
        "Demand": info["demand"],
        "Rooms Sold": info["rooms_sold"],
        "Occupancy (%)": occupancy,
        "Revenue": total_revenue
    })

    state = next_state
    day_number += 1

In [4]:
trajectory_df = pd.DataFrame(trajectory)

trajectory_df.head()

,Day,Price,Demand,Rooms Sold,Occupancy (%),Revenue
0,1,140,1,1,2.0,115
1,2,140,0,0,2.0,115
2,3,140,0,0,2.0,115
3,4,140,0,0,2.0,115
4,5,140,0,0,2.0,115


In [5]:
trajectory_df.to_csv("../data/processed/price_trajectory.csv",index=False)

print("price_trajectory.csv saved successfully.")

price_trajectory.csv saved successfully.
